# Whatdog Pipeline
## Data Pipeline

In [ ]:
import os
import data_pipeline as dp
import model_pipeline as mp
import torch

In [ ]:
# Get directory of images
current_dir = os.getcwd()
root_dir = current_dir + "/Images"

In [ ]:
# Load the dog dataset using torchvision.datasets.ImageFolder
dog_dataset = dp.load_imagefolder_dataset(root_dir)

# Obtain the transformations for training and test/val using Imagenet1k's mean and std
train_transform, val_test_transform = dp.get_transformations(
    mean=[0.485, 0.456, 0.406], 
    std=[0.229, 0.224, 0.225]
)

# Split the dataset into train(70%)/val(15%)/test(15%)
train_dataset, val_dataset, test_dataset = dp.split_dataset(dog_dataset)

In [ ]:
label_mapping = dp.clean_label_mapping(dog_dataset)

In [ ]:
# Wrap the subsets in a new subset class to attach the image transformations
train_dataset = dp.TransformedSubset(train_dataset, train_transform)
val_dataset = dp.TransformedSubset(val_dataset, val_test_transform)
test_dataset = dp.TransformedSubset(test_dataset, val_test_transform)

In [ ]:
train_batch_size = 128
val_batch_size = 256

In [ ]:
train_loader, val_loader, test_loader = dp.create_dataloaders(
    train_dataset=train_dataset, 
    val_dataset=val_dataset, 
    test_dataset=test_dataset, 
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size, 
    test_batch_size=val_batch_size, 
    train_shuffle=True
)

In [ ]:
random_train_label = dp.load_one_image(train_loader).item()

print(f"Label: {label_mapping[random_train_label]}")
print(f"Integer Label: {random_train_label}")

In [ ]:
device = mp.get_device()

In [ ]:
model = mp.load_and_finetune_resnet18()

In [ ]:
learning_rate = 1e-5

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate,
    weight_decay=0.01
)

In [ ]:
mp.training_loop(
    model=model,
    train_loader=train_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    num_epochs=25,
    device=device
)

In [ ]:
# CODE FOR LOADING FROM A .pt FILE

# pt_file_name = ""
# pt_model = mp.load_and_finetune_resnet18()
# loss_fn = torch.nn.CrossEntropyLoss()
# # Reinitialize the optimizer that was used.
# optimizer = torch.optim.AdamW(
#     filter(lambda p: p.requires_grad, model.parameters()),
#     lr=1e-3,
#     weight_decay=0.01
# )
# model_from_pt = mp.load_model_from_pt(pt_model, optimizer, pt_file_name)

In [ ]:
mp.evaluate_accuracy(model=model, val_loader=train_loader, device=device)

In [ ]:
mp.evaluate_accuracy(model=model, val_loader=val_loader, device=device)

In [ ]:
mp.save_model(
    model=model,
    optimizer=optimizer, 
    epochs_trained=35, 
    learning_rate=learning_rate,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    accuracy=75.74, 
    note="Tried lr=1e-5 and a larger batch size. The model is still overfitting. Could be a number of other issues.", 
    save_file_path="trained_models/resnet18-ft-layer4-classification-layer-75-percent-acc-larger-batch-size"
)